Parse all dates, normalise them with AI

For external API
Use external API to get lat and long for the place where the event happend. That way all the locations will be normalized 

Fix Sex Column - 11 Genders! Its 2026, all right
Fix Age Column - 251 unique value, WTF!
Fix Country - Maybe directly with API - 252 value, a bit too much 
Fix Type - Some are dublicated

Fix all unknown, sometimes they are Unknown, sometimes ? and sometimes Nan

Fix Date

In [192]:
%pip install dateparser

Note: you may need to restart the kernel to use updated packages.


In [193]:
import pandas as pd
import dateparser 
import seaborn as sns

path = "GSAF5_fix.xlsx"
pd.set_option("display.max_columns", None)

In [194]:
data = pd.read_excel(path)
# keep original text for reference
data["Date_orig"] = data["Date"]

import re
from datetime import datetime, timedelta


def parse_date_info(x, year=None):
    """Parse a raw date string and return a dict with normalization

    ``year`` is an optional secondary year value (from the ``Year`` column).
    When the text lacks a four‑digit year, this value is used to fill in the
    missing component (e.g. "29th January" with Year=1999).

    - ``normalized``: a datetime representing the best guess for the event
      (used to populate ``Date`` column later).
    - ``is_exact_date``: ``True`` only if day, month and year are present in
      the original string.
    - ``date_from`` / ``date_to``: for ranges/periods.
    - ``date_extra``: any removed/annotated text ("reported", "before ...",
      etc.) or a note for unparsed values.
    """
    info = {
        "normalized": None,
        "is_exact_date": False,
        "date_from": None,
        "date_to": None,
        "date_extra": None,
    }
    if pd.isna(x):
        return info
    s = str(x).strip()

    # pull year from separate column if available; will be used later
    orig_year = None
    if year is not None and not pd.isna(year):
        try:
            orig_year = int(year)
        except Exception:
            orig_year = None

    # quick negative cases
    if re.match(r'(?i)^(no\s*date|none|nan)\b', s):
        info["date_extra"] = "no date"
        return info

    # housekeeping / normalization of input text ------------------------------------------------
    # drop leading and trailing junk
    s = s.rstrip(". ")
    # drop a leading "Reported" but record that fact
    if s.lower().startswith("reported"):
        s = s[len("reported"):].strip()
        info["date_extra"] = "reported"

    # remove stray letters after a valid date ("19-Jul-2007.b")
    s = re.sub(r"(\d{1,2}-[A-Za-z]{3}-\d{2,4})[\.][A-Za-z]$", r"\1", s)

    # convert various separators to plain spaces for easier matching
    s = s.replace("–", "-")
    s = s.replace("/", " ")

    # handle obvious OCR/truncation issues
    # e.g. "190Feb-2010" -> "19 Feb-2010"
    s = re.sub(r"(?i)\b(\d{3})([A-Za-z]+)\b", lambda m: m.group(1)[:2] + " " + m.group(2), s)

    # mid/early/late prefixes that dateparser can't handle
    m_phase = re.match(r"(?i)^(early|mid|late)\s+([A-Za-z]+)[\s\-](\d{4})", s)
    if m_phase:
        phase = m_phase.group(1).lower()
        month_name = m_phase.group(2)
        year = int(m_phase.group(3))
        # try parsing month name generically
        try:
            tmp = dateparser.parse(f"1 {month_name}")
            month = tmp.month if tmp else 1
        except Exception:
            month = 1
        if phase == "early":
            day = 5
        elif phase == "mid":
            day = 15
        else:
            # late
            # pick 28 to account for February or months with 30/31
            day = 28
        info["normalized"] = datetime(year, month, day)
        info["is_exact_date"] = False
        info["date_extra"] = phase
        return info

    # "Late Jul-2008" -> we'll just drop the "Late" and later assign day=28
    if re.match(r"(?i)^late\b", s):
        s = re.sub(r"(?i)^late\s+", "", s)
        # treat in date_extra so we know it was approximate
        info["date_extra"] = ((info.get("date_extra") or "") + " late").strip()

    # seasons that dateparser doesn't handle directly
    season_map = {
        "spring": (4, 15),
        "summer": (7, 15),
        "fall": (10, 15),
        "autumn": (10, 15),
        "winter": (1, 15),
    }
    for season, (month, day) in season_map.items():
        if re.match(rf"(?i)^{season}[\s\-](\d{{4}})", s):
            year = int(re.search(r"(\d{4})", s).group(1))
            info["normalized"] = datetime(year, month, day)
            info["is_exact_date"] = False
            info["date_extra"] = season
            return info

    # explicit periods like ``1845-1853`` or ``1845 – 1853``
    m_range = re.match(r'^(?P<start>\d{4})\s*[-–]\s*(?P<end>\d{4})$', s)
    if m_range:
        y1 = int(m_range.group("start"))
        y2 = int(m_range.group("end"))
        dt1 = datetime(y1, 1, 1)
        dt2 = datetime(y2, 12, 31)
        info["date_from"] = dt1
        info["date_to"] = dt2
        info["normalized"] = dt1 + (dt2 - dt1) / 2
        info["is_exact_date"] = False
        info["date_extra"] = f"period {y1}-{y2}"
        return info

    # "Between 1918 & 1939" style ranges
    m_between = re.match(r'(?i)^between\s*(?P<start>\d{4})\s*[&and]+\s*(?P<end>\d{4})', s)
    if m_between:
        y1 = int(m_between.group("start"))
        y2 = int(m_between.group("end"))
        dt1 = datetime(y1, 1, 1)
        dt2 = datetime(y2, 12, 31)
        info["date_from"] = dt1
        info["date_to"] = dt2
        info["normalized"] = dt1 + (dt2 - dt1) / 2
        info["is_exact_date"] = False
        info["date_extra"] = f"between {y1}-{y2}"
        return info

    # "Before <date>" case (year or full date)
    m_before = re.match(r'(?i)^before\s*(.+)', s)
    if m_before:
        stripped = m_before.group(1).strip()
        # try to parse the stripped portion normally
        dt = dateparser.parse(stripped)
        if dt:
            info["date_to"] = dt
            # choose a normalized point halfway between start of era and dt
            info["normalized"] = dt - timedelta(days=30)
            info["is_exact_date"] = False
            info["date_extra"] = f"before {dt.date()}"
            return info
        else:
            # fallback to only capturing year
            year_match = re.match(r"(\d{4})", stripped)
            if year_match:
                year = int(year_match.group(1))
                info["date_to"] = datetime(year, 1, 1)
                info["normalized"] = datetime(year, 6, 30)
                info["is_exact_date"] = False
                info["date_extra"] = f"before {year}"
                return info

    # handle decades like "1960s" or "1960's" explicitly
    m_decade = re.match(r"^(?P<dec>\d{3}0)'?s$", s)
    if m_decade:
        decade = int(m_decade.group("dec"))
        if orig_year and decade <= orig_year < decade + 10:
            year_choice = orig_year
        else:
            year_choice = decade + 5
        info["normalized"] = datetime(year_choice, 6, 15)
        info["is_exact_date"] = False
        info["date_extra"] = "decade"
        return info

    # final fallback: let dateparser try to do the heavy lifting
    dt = dateparser.parse(s)
    if dt:
        # if year absent from string but available separately, inject it
        if orig_year and not re.search(r'\b\d{4}\b', s):
            try:
                dt = dt.replace(year=orig_year)
            except Exception:
                pass
        info["normalized"] = dt
        # crude check for presence of day and month in original text
        has_day = bool(re.search(r'\b([0-3]?\d)\b', s)) and not bool(
            re.match(r'^\d{4}$', s)
        )
        has_month = bool(
            re.search(r'(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec|\d{1,2}[\s\-]\d{1,2})', s, re.I)
        )
        info["is_exact_date"] = has_day and has_month

        # after parsing, check for weekend mention – shift back a day
        if re.search(r'weekend', s, re.I) and info["normalized"]:
            info["normalized"] = info["normalized"] - timedelta(days=1)
            info["date_extra"] = (info.get("date_extra") or "") + " weekend"
        # handle approximations
        if info["is_exact_date"] is False:
            if "late" in (info.get("date_extra") or ""):
                info["normalized"] = info["normalized"].replace(day=28)
            elif "early" in (info.get("date_extra") or ""):
                info["normalized"] = info["normalized"].replace(day=5)
            elif "mid" in (info.get("date_extra") or ""):
                info["normalized"] = info["normalized"].replace(day=15)
        return info

    # nothing could be parsed
    info["date_extra"] = "unparsed"
    return info


# apply parser row-wise so the Year column can be used when the string lacks a year
parsed = data.apply(lambda r: parse_date_info(r["Date_orig"], r.get("Year")), axis=1)
parsed_df = pd.DataFrame(parsed.tolist(), index=data.index)

# merge new columns into the main frame
data = pd.concat([data, parsed_df], axis=1)

# post-processing fixes --------------------------------------------------
# 1. rows where year was truncated (e.g. "10-Jul-202"); borrow year from next
mask_short_year = data["Date_orig"].astype(str).str.match(r"\d{1,2}-[A-Za-z]{3}-\d{1,3}$")
for idx in data[mask_short_year & data["normalized"].isna()].index:
    nxt = idx + 1
    if nxt in data.index and pd.notna(data.loc[nxt, "normalized"]):
        year = data.loc[nxt, "normalized"].year
        orig = data.loc[idx, "Date_orig"]
        corrected = re.sub(r"(\d{1,2}-[A-Za-z]{3}-)(\d{1,3})$", lambda m: m.group(1) + str(year), orig)
        repair = parse_date_info(corrected)
        for key, val in repair.items():
            data.at[idx, key] = val
        if repair["normalized"]:
            data.at[idx, "Date"] = repair["normalized"].strftime("%Y-%m-%d")

# 2. ensure "date_to" is set for before cases even if normalized was filled earlier
before_mask = data["Date_orig"].astype(str).str.lower().str.startswith("before")
for idx in before_mask.index:
    if pd.isna(data.at[idx, "date_to"]) and pd.notna(data.at[idx, "normalized"]):
        data.at[idx, "date_to"] = data.at[idx, "normalized"]

# overwrite the original "Date" column with ISO repr of normalized value
# (it may already have been set by first pass, but redo to catch repairs)
data["Date"] = data["normalized"].dt.strftime("%Y-%m-%d")

# capture rows that still failed to produce a normalized datetime
suspicious_dates_idx = data[data["normalized"].isna()].index
pd.DataFrame(suspicious_dates_idx).to_csv("idx.csv")


In [195]:
# 1/ Define helper to fill Year from normalized when Year == 0.0

def fill_year_from_normalized(df, year_col='Year', normalized_col='normalized'):
    """Fill missing/zero years in `year_col` using the year from `normalized_col`.

    Args:
        df: pandas.DataFrame containing both columns.
        year_col: name of the Year column (may be float like 0.0).
        normalized_col: name of the column with datetimes.

    Returns:
        The same DataFrame with updated year values (in-place).
    """
    # safety checks
    if year_col not in df.columns or normalized_col not in df.columns:
        return df

    # consider zero or exact 0.0 as missing; also treat NaN
    try:
        mask_zero = (df[year_col] == 0.0)
    except Exception:
        # fallback if non-numeric
        mask_zero = df[year_col].astype(str).isin(['0', '0.0'])

    mask_na = df[year_col].isna()
    mask = (mask_zero | mask_na) & df[normalized_col].notna()

    if mask.any():
        df.loc[mask, year_col] = df.loc[mask, normalized_col].dt.year
    return df


# 2/ Apply the helper to the main dataframe
data = fill_year_from_normalized(data, year_col='Year', normalized_col='normalized')

# quick check
print('Filled Year from normalized for', int(((data['Year'] == 0) | (data['Year'].isna())).sum()), 'rows remain with Year==0 or NaN')


Filled Year from normalized for 41 rows remain with Year==0 or NaN


In [196]:
# Normalize column names: lowercase, only letters/digits/underscore, spaces -> underscore
import re

def normalize_columns(df):
    """Normalize DataFrame column names:
    - lower case
    - spaces and other non-alnum -> underscore
    - collapse multiple underscores
    - strip leading/trailing underscores
    """
    new_cols = []
    for col in df.columns:
        s = str(col).strip().lower()
        # replace non-alphanumeric with underscore
        s = re.sub(r'[^0-9a-zA-Z]+', '_', s)
        # collapse multiple underscores
        s = re.sub(r'_+', '_', s)
        # strip leading/trailing underscores
        s = s.strip('_')
        # ensure not empty
        if s == '':
            s = 'col'
        new_cols.append(s)
    df.columns = new_cols
    return df

# Apply normalization to `data`
data = normalize_columns(data)
print('Columns after normalization:', data.columns.tolist())


Columns after normalization: ['date', 'year', 'type', 'country', 'state', 'location', 'activity', 'name', 'sex', 'age', 'injury', 'fatal_y_n', 'time', 'species', 'source', 'pdf', 'href_formula', 'href', 'case_number', 'case_number_1', 'original_order', 'unnamed_21', 'unnamed_22', 'date_orig', 'normalized', 'is_exact_date', 'date_from', 'date_to', 'date_extra']


In [197]:
data = data.dropna(subset=["normalized"]) # Dropping all rows that I am too lazy to parse

In [198]:
data.info()


<class 'pandas.DataFrame'>
Index: 6966 entries, 0 to 7073
Data columns (total 29 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   date            6966 non-null   str           
 1   year            6966 non-null   float64       
 2   type            6950 non-null   str           
 3   country         6918 non-null   str           
 4   state           6501 non-null   str           
 5   location        6426 non-null   str           
 6   activity        6406 non-null   str           
 7   name            6757 non-null   str           
 8   sex             6402 non-null   str           
 9   age             4067 non-null   object        
 10  injury          6933 non-null   str           
 11  fatal_y_n       6413 non-null   object        
 12  time            3539 non-null   object        
 13  species         3905 non-null   str           
 14  source          6946 non-null   object        
 15  pdf             6691

In [207]:
data.head()

,date,year,type,country,state,location,activity,name,sex,age,injury,fatal_y_n,time,species,source,pdf,href_formula,href,case_number,case_number_1,original_order,unnamed_21,unnamed_22,date_orig,normalized,is_exact_date,date_from,date_to,date_extra
0,2026-01-29,2026.0,Unprovoked,Brazil,Recife,Del Chifre Beach in Olinda,Swimming,Deivson Rocha Dantas,M,13,Right thigh and lower leg stripped of flesh,Y,?,Unknown bull and tiger sharks frequent the area,Kevin McMurray Trackingsharks.com: TV Globo: P...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,29th January,2026-01-29,False,NaT,2026-01-29,NaN
1,2026-01-29,2026.0,Unprovoked,Australia,NSW,Angels Beach East Ballina,Surfing,Unnamed man,M,?,No injury shark knocked man of his board,N,1100hrs,Unknown,Bob Myatt GSAF,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,29th January,2026-01-29,False,NaT,2026-01-29,NaN
2,2026-01-24,2026.0,Unprovoked,Australia,Tasmania,Cooee Beach west of Burnie,Swimming,Megan Stokes,F,?,Puncture wounds to right knee,N,1815hrs,1.7m Seven Gill shark,Bob Myatt GSAF,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24th January,2026-01-24,False,NaT,2026-01-24,NaN
3,2026-01-20,2026.0,Unprovoked,Australia,NSW,Point Plomber North of Port Macquarie,Surfing,Paul Zvirdinas,M,39,Minor cuts and abrasions,N,0830hrs,Bull shark,Bob Myatt GSAF,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20th January,2026-01-20,False,NaT,2026-01-20,NaN
4,2026-01-19,2026.0,Unprovoked,Australia,NSW,Dee Why,Surfing,Unknown,M,11,None reported damage to board,N,1145hrs,Bull shark,Andy Currie,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,19th January,2026-01-19,False,NaT,2026-01-19,NaN


In [202]:
# Interactive plot with plotly.express
import plotly.express as px
import datetime

# ensure `normalized` is datetime
if not pd.api.types.is_datetime64_any_dtype(data['normalized']):
    data['normalized'] = pd.to_datetime(data['normalized'], errors='coerce')

# prepare data
data_plot = data.dropna(subset=['normalized']).copy()

df_counts = (
    data_plot['normalized'].dt.year
    .value_counts()
    .rename_axis('year')
    .reset_index(name='incidents')
    .sort_values('year')
)

# filter to sensible range: from 1800 to current year
current_year = datetime.datetime.now().year
df_counts = df_counts[df_counts['year'].between(1800, current_year)]

# build interactive Plotly figure
fig = px.line(
    df_counts,
    x='year',
    y='incidents',
    markers=True,
    title='Shark incidents per year (based on normalized date)',
    labels={'year': 'Year', 'incidents': 'Number of incidents'},
    template='plotly_white'
)
fig.update_traces(mode='lines+markers', hovertemplate='Year: %{x}<br>Incidents: %{y}')
fig.update_layout(xaxis=dict(range=[1800, current_year], dtick=10), height=600)
fig.show()


Who shark prefer to bite: Male or Female

In [203]:
data.groupby(data["normalized"].dt.year).size()

normalized
1018      1
1543      1
1554      1
1555      1
1595      1
       ... 
2022     98
2023    109
2024     54
2025     66
2026     11
Length: 257, dtype: int64